# Brain Tumor MRI Dataset - Exploratory Data Analysis

This notebook explores the Brain Tumor MRI image dataset before model training. It covers dataset structure, class balance, representative samples, image dimensions, preprocessing effects, augmentation examples, and practical observations for a PyTorch training pipeline.

**Expected local layout:** `data/archive (1)/Training/<class_name>/image.jpg` and `data/archive (1)/Testing/<class_name>/image.jpg`.

## Notebook Checklist

- Dataset overview
- Class distribution
- Sample image visualization
- Image dimensions analysis
- Preprocessing visualization
- Augmentation examples
- Written observations and modeling implications

## 1. Setup

In [ ]:
from pathlib import Path
from collections import Counter
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ModuleNotFoundError:
    sns = None

try:
    from PIL import Image, ImageOps, ImageEnhance, UnidentifiedImageError
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'Pillow is required for this notebook. Install project dependencies with: pip install -r requirements.txt'
    ) from exc

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25
plt.rcParams['font.size'] = 11

if sns is not None:
    sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / 'data').exists():
        return cwd
    if (cwd.parent / 'data').exists():
        return cwd.parent
    return cwd


PROJECT_ROOT = find_project_root()

candidate_roots = [
    PROJECT_ROOT / 'data' / 'archive (1)',
    PROJECT_ROOT / 'data' / 'Brain Tumor MRI Dataset',
    PROJECT_ROOT / 'data' / 'brain_tumor_mri',
    PROJECT_ROOT / 'data' / 'raw' / 'archive (1)',
    PROJECT_ROOT / 'data' / 'raw' / 'Brain Tumor MRI Dataset',
    PROJECT_ROOT / 'data' / 'raw',
]

DATASET_ROOT = next(
    (path for path in candidate_roots if (path / 'Training').exists() and (path / 'Testing').exists()),
    None,
)

if DATASET_ROOT is None:
    checked = '\n'.join(str(path) for path in candidate_roots)
    raise FileNotFoundError(
        'Could not find a Brain Tumor MRI dataset root with Training and Testing folders.\n'
        f'Checked:\n{checked}'
    )

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
SPLIT_ORDER = ['Training', 'Testing']

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset root: {DATASET_ROOT}')

## 2. Build Image Inventory

In [ ]:
records = []

for split in SPLIT_ORDER:
    split_dir = DATASET_ROOT / split
    for class_dir in sorted(path for path in split_dir.iterdir() if path.is_dir()):
        for image_path in sorted(class_dir.rglob('*')):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                records.append(
                    {
                        'split': split,
                        'class': class_dir.name,
                        'path': image_path,
                        'filename': image_path.name,
                        'extension': image_path.suffix.lower(),
                    }
                )

df = pd.DataFrame(records)

if df.empty:
    raise ValueError(f'No image files found under {DATASET_ROOT}')

df['split'] = pd.Categorical(df['split'], categories=SPLIT_ORDER, ordered=True)
df['class'] = df['class'].astype('category')

display(df.head())
print(f'Total images indexed: {len(df):,}')

## 3. Dataset Overview

In [ ]:
overview = pd.DataFrame(
    {
        'metric': [
            'dataset_root',
            'total_images',
            'splits',
            'classes',
            'file_extensions',
        ],
        'value': [
            str(DATASET_ROOT),
            f'{len(df):,}',
            ', '.join(df['split'].astype(str).unique()),
            ', '.join(sorted(df['class'].astype(str).unique())),
            ', '.join(sorted(df['extension'].unique())),
        ],
    }
)

split_summary = (
    df.groupby('split', observed=True)
    .size()
    .rename('image_count')
    .reset_index()
)

class_summary = (
    df.groupby('class', observed=True)
    .size()
    .rename('image_count')
    .sort_values(ascending=False)
    .reset_index()
)

display(overview)
display(split_summary)
display(class_summary)

### Written Observations - Dataset Overview

- The current local dataset contains **7,200 MRI images** across four labels: `glioma`, `meningioma`, `notumor`, and `pituitary`.
- The dataset is already separated into `Training` and `Testing`, with **5,600 training images** and **1,600 testing images**.
- The testing split should be treated as a final holdout set. If validation is needed, create it from the training split only.

## 4. Class Distribution

In [ ]:
distribution = (
    df.groupby(['split', 'class'], observed=True)
    .size()
    .rename('image_count')
    .reset_index()
)

display(distribution.pivot(index='class', columns='split', values='image_count').fillna(0).astype(int))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

if sns is not None:
    sns.barplot(data=distribution, x='class', y='image_count', hue='split', ax=axes[0])
    sns.barplot(data=class_summary, x='class', y='image_count', ax=axes[1])
else:
    distribution.pivot(index='class', columns='split', values='image_count').plot(kind='bar', ax=axes[0])
    axes[1].bar(class_summary['class'].astype(str), class_summary['image_count'])

axes[0].set_title('Class Distribution by Split')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Image Count')
axes[0].tick_params(axis='x', rotation=20)

axes[1].set_title('Overall Class Distribution')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Image Count')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

### Written Observations - Class Distribution

- Each class has **1,400 training images** and **400 testing images**, so the local copy is perfectly balanced at the class level.
- Because the classes are balanced, initial training does not require class weighting. Still, per-class recall and confusion matrices should be monitored because medical errors are not equally costly.
- A balanced dataset makes accuracy easier to interpret, but clinical model evaluation should still prioritize sensitivity, specificity, macro F1, and per-class performance.

## 5. Sample Image Visualization

In [ ]:
def load_rgb(path: Path) -> Image.Image:
    return Image.open(path).convert('RGB')


def plot_samples(dataframe: pd.DataFrame, split: str, samples_per_class: int = 4) -> None:
    classes = sorted(dataframe['class'].astype(str).unique())
    fig, axes = plt.subplots(len(classes), samples_per_class, figsize=(4 * samples_per_class, 4 * len(classes)))

    if len(classes) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_idx, class_name in enumerate(classes):
        class_rows = dataframe[(dataframe['split'].astype(str) == split) & (dataframe['class'].astype(str) == class_name)]
        sample_rows = class_rows.sample(n=min(samples_per_class, len(class_rows)), random_state=RANDOM_STATE)

        for col_idx in range(samples_per_class):
            ax = axes[row_idx, col_idx]
            ax.axis('off')

            if col_idx >= len(sample_rows):
                continue

            item = sample_rows.iloc[col_idx]
            image = load_rgb(item['path'])
            ax.imshow(image, cmap='gray')
            ax.set_title(f'{split} | {class_name}\n{image.size[0]} x {image.size[1]}', fontsize=10)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_samples(df, split='Training', samples_per_class=4)

In [ ]:
plot_samples(df, split='Testing', samples_per_class=4)

### Written Observations - Sample Images

- Images appear to be brain MRI slices with a mix of tumor-positive categories and a `notumor` category.
- Visual differences can include tumor location, intensity patterns, scan framing, and brain orientation. A model may learn useful anatomy-related features, but it can also learn shortcuts from acquisition artifacts or image borders.
- The samples suggest that resizing and intensity normalization are necessary before batching images in PyTorch.

## 6. Image Dimensions Analysis

In [ ]:
dimension_records = []
corrupt_files = []

for item in df.to_dict('records'):
    try:
        with Image.open(item['path']) as image:
            width, height = image.size
            mode = image.mode
    except (UnidentifiedImageError, OSError) as exc:
        corrupt_files.append({'path': item['path'], 'error': str(exc)})
        continue

    dimension_records.append(
        {
            'split': item['split'],
            'class': item['class'],
            'path': item['path'],
            'width': width,
            'height': height,
            'mode': mode,
            'aspect_ratio': width / height,
            'megapixels': (width * height) / 1_000_000,
        }
    )

dims_df = pd.DataFrame(dimension_records)
corrupt_df = pd.DataFrame(corrupt_files)

display(dims_df.head())
print(f'Images with readable dimensions: {len(dims_df):,}')
print(f'Unreadable or corrupt files: {len(corrupt_df):,}')

In [ ]:
dimension_summary = dims_df[['width', 'height', 'aspect_ratio', 'megapixels']].describe().T

top_dimensions = (
    dims_df.groupby(['width', 'height'], observed=True)
    .size()
    .rename('image_count')
    .sort_values(ascending=False)
    .head(15)
    .reset_index()
)

dimension_by_class = (
    dims_df.groupby('class', observed=True)
    .agg(
        image_count=('path', 'count'),
        min_width=('width', 'min'),
        max_width=('width', 'max'),
        min_height=('height', 'min'),
        max_height=('height', 'max'),
        median_width=('width', 'median'),
        median_height=('height', 'median'),
    )
    .reset_index()
)

display(dimension_summary)
display(top_dimensions)
display(dimension_by_class)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if sns is not None:
    sns.histplot(data=dims_df, x='width', hue='split', bins=30, kde=False, ax=axes[0])
    sns.histplot(data=dims_df, x='height', hue='split', bins=30, kde=False, ax=axes[1])
    sns.scatterplot(data=dims_df, x='width', y='height', hue='class', alpha=0.55, ax=axes[2])
else:
    axes[0].hist(dims_df['width'], bins=30)
    axes[1].hist(dims_df['height'], bins=30)
    axes[2].scatter(dims_df['width'], dims_df['height'], alpha=0.55)

axes[0].set_title('Width Distribution')
axes[1].set_title('Height Distribution')
axes[2].set_title('Width vs Height')

for ax in axes:
    ax.set_xlabel(ax.get_xlabel() or 'Pixels')

plt.tight_layout()
plt.show()

### Written Observations - Image Dimensions

- The dataset should not be assumed to have one uniform image size. The local sample contains many 512 x 512 images, but some images are smaller or non-square.
- PyTorch training will require a fixed tensor shape, so images should be resized, center-cropped, or padded consistently.
- For medical imaging, resizing can affect small lesions. Prefer a deliberate target size such as 224 x 224 for baseline CNNs, then compare against higher-resolution training if compute allows.
- Any unreadable files listed by `corrupt_df` should be removed or repaired before training.

## 7. Preprocessing Visualization

In [ ]:
TARGET_SIZE = (224, 224)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def preprocess_for_cnn(path: Path, target_size: tuple[int, int] = TARGET_SIZE):
    original = Image.open(path).convert('RGB')
    resized = original.resize(target_size, resample=Image.BILINEAR)
    array = np.asarray(resized).astype(np.float32) / 255.0
    normalized = (array - IMAGENET_MEAN) / IMAGENET_STD
    display_ready = np.clip((normalized * IMAGENET_STD) + IMAGENET_MEAN, 0, 1)
    return original, resized, normalized, display_ready


sample_row = df[df['split'].astype(str) == 'Training'].sample(1, random_state=RANDOM_STATE).iloc[0]
original, resized, normalized, display_ready = preprocess_for_cnn(sample_row['path'])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(original, cmap='gray')
axes[0].set_title(f'Original\n{original.size[0]} x {original.size[1]}')
axes[0].axis('off')

axes[1].imshow(resized, cmap='gray')
axes[1].set_title(f'Resized\n{TARGET_SIZE[0]} x {TARGET_SIZE[1]}')
axes[1].axis('off')

axes[2].imshow(display_ready)
axes[2].set_title('Normalized Tensor\nshown after de-normalization')
axes[2].axis('off')

plt.suptitle(f'Preprocessing Example: {sample_row["class"]}', y=1.03)
plt.tight_layout()
plt.show()

channel_stats = pd.DataFrame(
    {
        'channel': ['red', 'green', 'blue'],
        'normalized_mean': normalized.mean(axis=(0, 1)),
        'normalized_std': normalized.std(axis=(0, 1)),
        'normalized_min': normalized.min(axis=(0, 1)),
        'normalized_max': normalized.max(axis=(0, 1)),
    }
)

display(channel_stats)

In [ ]:
original_gray = np.asarray(original.convert('L')).astype(np.float32) / 255.0
resized_gray = np.asarray(resized.convert('L')).astype(np.float32) / 255.0

plt.figure(figsize=(12, 5))
plt.hist(original_gray.ravel(), bins=50, alpha=0.55, label='original grayscale')
plt.hist(resized_gray.ravel(), bins=50, alpha=0.55, label='resized grayscale')
plt.title('Intensity Distribution Before and After Resizing')
plt.xlabel('Intensity')
plt.ylabel('Pixel Count')
plt.legend()
plt.tight_layout()
plt.show()

### Written Observations - Preprocessing

- Baseline preprocessing converts images to RGB, resizes to 224 x 224, scales pixel values to `[0, 1]`, and normalizes channels with ImageNet statistics.
- RGB conversion is useful for pretrained CNN backbones such as ResNet or EfficientNet, even when the source MRI appears grayscale.
- If training a model from scratch, dataset-specific mean and standard deviation may be preferable to ImageNet statistics.
- Avoid preprocessing choices that hide clinically relevant low-contrast structures. Always inspect before and after examples.

## 8. Augmentation Examples

In [ ]:
def mild_mri_augmentation(image: Image.Image, rng: random.Random) -> Image.Image:
    augmented = image.copy()

    if rng.random() < 0.5:
        augmented = ImageOps.mirror(augmented)

    angle = rng.uniform(-10, 10)
    augmented = augmented.rotate(angle, resample=Image.BILINEAR, fillcolor=(0, 0, 0))

    contrast_factor = rng.uniform(0.90, 1.15)
    brightness_factor = rng.uniform(0.90, 1.10)
    augmented = ImageEnhance.Contrast(augmented).enhance(contrast_factor)
    augmented = ImageEnhance.Brightness(augmented).enhance(brightness_factor)

    return augmented


base_row = df[df['split'].astype(str) == 'Training'].sample(1, random_state=RANDOM_STATE + 7).iloc[0]
base_image = Image.open(base_row['path']).convert('RGB').resize(TARGET_SIZE, resample=Image.BILINEAR)
local_rng = random.Random(RANDOM_STATE)

augmented_images = [base_image] + [mild_mri_augmentation(base_image, local_rng) for _ in range(7)]
titles = ['Original'] + [f'Augmented {idx}' for idx in range(1, 8)]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, image, title in zip(axes.ravel(), augmented_images, titles):
    ax.imshow(image, cmap='gray')
    ax.set_title(title)
    ax.axis('off')

plt.suptitle(f'Mild Augmentation Examples: {base_row["class"]}', y=1.02)
plt.tight_layout()
plt.show()

### Written Observations - Augmentation

- Mild rotations, small brightness and contrast changes, and cautious horizontal flips can help improve generalization.
- Aggressive rotation, vertical flips, heavy cropping, or strong geometric distortion may create anatomically implausible MRI examples.
- Augmentation policies should be validated with domain knowledge. For clinical images, preserving anatomical orientation and lesion visibility matters more than maximizing visual variety.
- Keep augmentation active only for training. Validation and testing should use deterministic preprocessing.

## 9. Final Written Observations and Modeling Implications

- **Dataset readiness:** The dataset is already organized for supervised classification with train and test folders. This is suitable for a PyTorch `ImageFolder` baseline.
- **Class balance:** The four classes are balanced in both training and testing, which simplifies early experiments and makes macro metrics easier to compare.
- **Validation strategy:** Reserve the provided testing split for final evaluation. Create a stratified validation split from `Training` for model selection and hyperparameter tuning.
- **Image shape:** Images are not guaranteed to share identical dimensions. Use a fixed preprocessing pipeline before batching.
- **Baseline preprocessing:** Start with RGB conversion, resize to 224 x 224, tensor conversion, and normalization. Revisit target resolution if tumor details appear too small after resizing.
- **Augmentation:** Use conservative augmentation. In medical imaging, unrealistic transforms can damage label validity.
- **Evaluation focus:** In addition to accuracy, report confusion matrix, macro F1, per-class precision and recall, sensitivity, and specificity.
- **Risk note:** This EDA does not establish clinical validity. Any diagnostic model trained from this dataset would require rigorous validation before real-world medical use.